# LLooM: Getting Started - Template Notebook

Last Updated: November 2024

### Installation
First, install the LLooM Python package, available on PyPI as [`text_lloom`](https://pypi.org/project/text_lloom/). We recommend setting up a virtual environment with [venv](https://docs.python.org/3/library/venv.html#creating-virtual-environments) or [conda](https://conda.io/projects/conda/en/latest/user-guide/tasks/manage-environments.html#creating-an-environment-with-commands).

In [2]:
!pip install text_lloom ollama sentence-transformers -q
!pip install "transformers<5.0.0" -q

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
datasets 3.6.0 requires dill<0.3.9,>=0.3.0, but you have dill 0.4.1 which is incompatible.
datasets 3.6.0 requires multiprocess<0.70.17, but you have multiprocess 0.70.19 which is incompatible.
chromadb 1.5.1 requires httpx>=0.27.0, but you have httpx 0.25.2 which is incompatible.


### Imports

In [29]:
import pandas as pd
import text_lloom.workbench as wb
from text_lloom.llm import Model, EmbedModel
import os
os.environ["TRANSFORMERS_NO_TORCHCODEC"] = "1"

pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_rows', None)

By default for quick setup, LLooM uses the OpenAI API under the hood to support its core operators (currently with GPT-4o and GPT-4o mini). You'll first need to locally set the `OPENAI_API_KEY` variable to use your own account. Alternatively, see our documentation on Custom Models to use different LLMs.

### Load data
For this example, we'll be using a sample dataset of 100 **Facebook posts** from **political** pages, gathered via CrowdTangle. The main columns we'll be using in our analysis are the following:
- `doc_id`: Unique ID for each post
- `text`: The text of the Facebook post
- `Page Category`: The category of the Facebook page
- `Likes`: The number of "likes" that the post received

In [ ]:
# We'll load data from an existing CSV
#data_link = "https://michelle123lam.github.io/lloom/data/political_fb_posts_100.csv"
#df = pd.read_csv(data_link)

df = pd.read_csv("../data/lloom_pilot_sample.csv")

In [10]:
# Preview of dataframe
display(df[["doc_id", "text"]].head())

doc_id  \
0   file7614   
1  file11640   
2  file10366   
3   file9125   
4   file2223   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                     

## v1: Manual mode

This notebook shows two example workflows: **v1: Manual mode**, or **v2: Auto mode**. We recommend starting with **v1: Manual mode** to survey the LLooM concepts and get a sense for the underlying functions.

### Create a LLooM instance
Then, after loading your data as a Pandas DataFrame, create a new LLooM instance. You will need to specify the name of the column that contains your input text documents (`text_col`). The ID column (`id_col`) is optional.

In [35]:
from text_lloom.llm import Model, EmbedModel

def setup_llm_fn(api_key):
    from openai import OpenAI
    return OpenAI(api_key="local", base_url="http://127.0.0.1:8080/v1")

def setup_embed_fn(api_key):
    from sentence_transformers import SentenceTransformer
    return SentenceTransformer("all-MiniLM-L6-v2")

async def call_llm_fn(model, prompt):
    if "system_prompt" not in model.args:
        model.args["system_prompt"] = (
            "You are a helpful assistant who helps with identifying patterns in text examples from 16th century letters written during the Reformation. Pay attention to theological ideas and arguments."
            "Always respond with raw JSON only — no markdown code fences, no explanation, no preamble."
            "When generating concepts, each concept MUST include ALL required fields: "
            "name, prompt, and example_ids (a list of document IDs, use an empty list if unsure)."
        )
    res = model.client.chat.completions.create(
        model=model.name,
        temperature=0,
        max_tokens=2048,
        messages=[
            {"role": "system", "content": model.args["system_prompt"]},
            {"role": "user", "content": prompt},
        ]
    )
    text = res.choices[0].message.content
    # Strip markdown fences if the model adds them anyway
    text = text.strip()
    if text.startswith("```"):
        text = text.split("```")[1]
        if text.startswith("json"):
            text = text[4:]
        text = text.strip()
    return text, [0, 0]

def call_embed_fn(model, text_arr):
    embeddings = model.client.encode(text_arr)
    return embeddings.tolist(), [0, 0]

api_key = "EMPTY"

In [32]:
MODEL_NAME = "gemma-4-26b-a4b-it"

l = wb.lloom(
    df=df,
    id_col="doc_id",
    text_col="text",
    distill_model=Model(
        setup_fn=setup_llm_fn, fn=call_llm_fn,
        name=MODEL_NAME, cost=[0, 0], rate_limit=(4, 1),
        context_window=32768, api_key=api_key,
    ),
    cluster_model=EmbedModel(
        setup_fn=setup_embed_fn, fn=call_embed_fn,
        name="", cost=0, batch_size=512, api_key=api_key,
    ),
    synth_model=Model(
        setup_fn=setup_llm_fn, fn=call_llm_fn,
        name=MODEL_NAME, cost=[0, 0], rate_limit=(2, 1),
        context_window=32768, api_key=api_key,
    ),
    score_model=Model(
        setup_fn=setup_llm_fn, fn=call_llm_fn,
        name=MODEL_NAME, cost=[0, 0], rate_limit=(4, 1),
        context_window=32768, api_key=api_key,
    ),
)

### Run concept generation
Next, you can go ahead and start the concept induction process by generating concepts. You can omit the `seed` parameter if you do not want to use a seed.

In [36]:
cur_seed = "theologically ideas and positions of the Reformation and their justification"
await l.gen(seed=cur_seed)

N sentences: Median=22, Std=15.34
Auto-suggested parameters: {'filter_n_quotes': 17, 'summ_n_bullets': 12, 'synth_n_concepts': 6}
Cost estimates not available for distill model `gemma-4-26b-a4b-it`
Cost estimates not available for cluster model ``
Cost estimates not available for synth model `gemma-4-26b-a4b-it`


Action required


Distill-filter
⠋ Loading Error Request timed out.
⠹ Loading ERROR json_load on: {
    "relevant_quotes": [
        "suscitati scilicet sunt a flabellis papisticis",
        "qui nihil aliud cogitant, quam quo pacto perdant evangelium una cum eius assertoribus",
        "qui nihil aliud cogitant, quam quo pacto perdant evangelium una cum eius assertoribus",
        "ne obolus quidem, nec satago; at fluere aliquid, si satagerem",
        "nihil aliud cogitant, quam quo pacto perdant evangelium una cum eius assertoribus",
        "nihil aliud cogitant, quam quo pacto perdant evangelium una cum eius assertoribus",
        "nihil aliud cogitant, quam quo pacto pe

,doc_id,text
0,file7614,"Gott mitt üch!\nGott wölle, das es ein guͦt end näme\nGott erbarme sich unser!\nGott mitt üch."
1,file11640,"das ir euch selbst ewres ampts drunder werdet wissen zuerinnen alle ding wol prueffenn, auch michts für so gewiß hallten, wenn gott mit der hschrifft zeugknug ettwas besserg oder hellers offennbarte\ninn was füisternus irrung und unverstand die gantze christen hait ain lange zeit gestanden\ndas auch das erkanntus unsers herren Jhesu Christi, bevorab, das nach dem h gaiste und seiner keinlischn Gloviend durch die saxhisterei der schulleerer und allenlai menstgliche gedancken und khunst, vil hundert jar, fur mehrertail verdunckellt\ndas wir in chottes forchte ymerie mehr wandellun, unnser. schwachtit wol enlernen unnd bedenncken, das die iurung aller flarische vonnatur angeborenn\nda er uns in seine gottliche wanhait mag beren\nda deme, denn henem 14 diß denn daas zu erbawung des leibes Christi gnedigelich hat offenmberet biß die heiligen gottes alle hinankommen zu aierlaz glauben und kanntnus des sunes gottes\ndas ein mennsch nichts nemen kan, es werd ichm denn vom hunell gegeben\ndas wir arme alles von Obenheraber durch Jhesum len messen erbeettellen unnd des henlischen reichtems gottes nichts gewis konnen habenn\nes sei denn unnser hertz und gewissen mit empfündtlicher lebendiger krafft gottes neben dem hellen zeügknug der schrifft dangei versichent und besigellt\ngottss offennbarung und diesalbung die alles leeret\nder artickell von der herrlichagi Christi unnd seiner h mennschait, der höchsten punc ain ers glaubens ist\ndas durch den glautenn ewre hertzen erfrewen unnd zu fiid soll stellenn\ndie gristerfaines gegs dempffen, noch dir prophecknnngen verachtenn\ndas die iurung aller flarische vonnatur angeborenn, wie es auch dem herren Jhesu vil gnad und anbeit braucht\nda deme, denn henem 14 diß denn daas zu erbawung des leibes Christi gnedigelich hat offenmberet\ndas die iurung aller flarische vonnatur angeborenn, wie es auch dem herren Jhesu vil gnad und anbeit braucht, damit er uns in seine gottliche wanhait mag beren\ndas wir in chottes forchte ymerie mehr wandellun, unnser. schwachtit wol enlernen unnd bedenncken"
2,file10366,"verharren ann gottes wort\nwie wol es deß handelß halb by u.f.g. stadt\ngott lob und danckend\nmitt gottes gnad getröst inn erckanter warheyt beston und fürfaren\nwie wir dann bißhar vil truͤbsal erlitten und noch, inn denen unß denocht gott trüwlich erhallten\neinmuͤtiger will und styfs fürnemmen, by angenomner warheyt mitt gottes gnad ze beharren\nmitt gottes gnad getröst\ninn erckanter warheyt beston\ngott trüwlich erhallten\nverharren ann gottes wort\ngott lob und danckend\nmitt gottes gnad getröst inn erckanter warheyt beston\nan gottes wort\ngott trüwlich erhallten\nangenomner warheyt mitt gottes gnad ze beharren\ngott beware sy\ndanckend u.f.g. deß früntlichen schrybens"
3,file9125,"magnatum tam nostrae quam papisticae ecclesiae maior pars suffragiis ad eum pro rege eligendum inclinant\nqui tot insontes heroas crudelissime trucidavit\nimmanitate illa tyrannica et ingenti strage in pios insontes crudeliter et fraudulenter patrata\npersuasi et conscientia retracti se a proposito paulum abduci passi sunt\nLegatus enim, qui est, episcopus, miris utitur excusationibus, quibus trucidatos misere calumniatur et culpam in illos totam transfert\nquasi molirentur regi perniciem et conspirati ad nuptias illas venissent\nfaventibus illi et calcar addentibus episcopis et similibus de grege porcis\nne tot falsis calumniis obruantur\nmiseremini non solum afflictae patriae nostrae sed etiam trucidatorum\nne tanto tam crudelis tyrannus honore afficiatur\nne tyrannus ille in hoc regno imperium occupet\nme miserum hominem, qui hic toti factioni exosus sum, exulem cum familiola habituri estis\nmagnatum tam nostrae quam papisticae ecclesiae maior pars\npiis insontes\nepiscopis et similibus de grege porcis\nfalsis calumniis obruantur\ntyrannus ille in hoc regno imperium occupet"
4,f



Distill-summarize
✅ Done    


,doc_id,text
0,file7614,Divine providence guides all human affairs
1,file7614,God remains the ultimate sovereign authority
2,file7614,Constant reliance on God's merciful grace
3,file7614,Prayer serves as essential spiritual connection
4,file7614,Seeking divine will in every outcome
5,file7614,Faithful submission to God's holy will
6,file7614,Recognition of God as supreme ruler
7,file7614,Constant hope in God's eternal mercy
8,file7614,Spiritual dependence on the creator alone
9,file7614,Blessings sought through constant divine intervention




Cluster
✅ Done    


,doc_id,text,cluster_id
0,file7614,Divine providence guides all human affairs,-1
360,file9380,Christian charity is a moral requirement,-1
359,file10093,"s ""Piety as a foundation for believers""",-1
358,file10093,Strengthening the mind in Christian faith,-1
357,file10093,Faithful recognition through Christ's holy name,-1
356,file10093,Bitter bile affecting the ungodly souls,-1
355,file10093,Wolves destroying sacred places through greed,-1
354,file10093,Steadfast struggle against worldly evil forces,-1
353,file10093,Sacrifice for the glory of Christ,-1
352,file10093,The alienation between ministers and heretics,-1




Synthesize
⠙ Loading Error Request timed out.
✅ Done    



Review
✅ Done    

    Auto-review:
    Removed (0):
        []
    Merged (0): 
    
    


Synthesize 1: (n=0 concepts)
✅ Done with concept generation!


### Review concepts

Review the generated concepts and select concepts to inspect further:

In [14]:
l.select()

In [15]:
# You can also double-check on your selected concepts with this command
l.show_selected()



Active concepts (n=4):
- Reformation Progress: Detect discussions on the slow advancement of reform.
- Scriptural Authority: Find mentions of scripture as ultimate authority.
- Clerical Reform: Locate calls for reforming church practices and clergy roles.
- Reformation Debates: Identify discussions on justification, works, and scriptural authority.


### Score concepts
Then, apply these concepts to the full dataset with `score()`. This function will score all documents with respect to each concept to indicate the extent to which the document matches the concept inclusion criteria.

In [16]:
# Run concept scoring
score_df = await l.score()

Cost estimates not available for score model `qwen2.5:7b-instruct-q4_K_M`


Action required
  0%|          | 0/4 [00:00<?, ?it/s]ERROR json_load on: {
    "pattern_results": [
        {
            "example_id": "file227",
            "rationale": "The text does not mention scripture or any form of ultimate authority.",
            "answer": "D",
            "quote": "Ecce C\u0119sorem.\nVil b\u00bcen und kuglen f\u00fcrt man uff , und ist der in trefflicher r\u00fcstung; wei\u0df nieman, wo \u0df."
        }
    ]
}
100%|██████████| 4/4 [42:46<00:00, 641.70s/it]   
✅ Done with concept scoring!


In [17]:
# View cost/time summary
l.summary()

Total time: 5720.24 sec (95.34 min)
	('Distill-filter', '2026-08-03-08-41-54'): 2410.05 sec
	('Distill-summarize', '2026-08-03-08-53-10'): 675.47 sec
	('Cluster', '2026-08-03-08-53-13'): 3.49 sec
	('Synthesize', '2026-08-03-08-54-03'): 49.97 sec
	('Review-remove', '2026-08-03-08-54-07'): 3.75 sec
	('Review-merge', '2026-08-03-08-54-18'): 10.69 sec
	('Score', '2026-08-03-19-31-50'): 2566.82 sec
Token and cost summaries not available for distill_model `qwen2.5:7b-instruct-q4_K_M`
Token and cost summaries not available for cluster_model ``
Token and cost summaries not available for synth_model `qwen2.5:7b-instruct-q4_K_M`
Token and cost summaries not available for score_model `qwen2.5:7b-instruct-q4_K_M`


### Visualize results
Now, you can visualize the results in the main **LLooM Workbench** view. An interactive widget will appear when you run the `vis` function:
![LLooM Workbench UI](https://github.com/michelle123lam/lloom/blob/main/docs/public/media/lloom_workbench_ui.png?raw=1)

The **Concept Overview (A)** provides a high-level summary. Click on a concept row in the **Concept Matrix (B)** to see its **Detail View (C)**, or click on a slice column to see its corresponding Detail View.

In [18]:
print(df.columns.tolist())
print(df.columns.duplicated().sum())

['doc_id', 'date', 'latin_text', 'enhg_text', 'rest_text', 'text', 'source', 'n_latin_sentences', 'n_enhg_sentences', 'n_rest_sentences']
0


In [19]:
# Visualize concept results
# Group data by the number of likes (automatically binned) with slice_col
l.vis()

In [ ]:
# Visualize concept results
# Group data by page category with slice_col
l.vis(slice_col="Page Category")

### (Optional) Try normalizing by slice or by concept


In [ ]:
l.vis(slice_col="Likes", norm_by="slice")

In [ ]:
l.vis(slice_col="Likes", norm_by="concept")

### (Optional) Add manual concept
You may also manually add your own custom concepts by providing a name and prompt. This will automatically score the data by that concept. Re-run the `vis()` function to see the new concept results.

In [ ]:
# Add a custom concept with the given name and prompt
await l.add(
    name="Your new concept name",
    prompt="Your new concept criteria prompt",  # Ex: "Does the text include [...]?"
)

In [ ]:
# Visualize concept results
l.vis(slice_col="Likes")

### (Optional) Submit your results
**🖼️ ✨ Submit your work for a chance to be featured on our site!**

If you'd like to share what you've done with LLooM or would like your work featured in a gallery of results, please submit your LLooM instance with the `submit()` function! If your submission is selected, we'll reach out to you to follow up and hear more about your work with LLooM.

In [ ]:
l.submit()  # You will be prompted to provide a few details about your analysis

### (Optional) Export and/or save results

In [20]:
# Export the results to a dataframe
export_df = l.export_df()

In [28]:
score_df[score_df["score"] >= 0.75]["doc_id"].value_counts().head(50)

doc_id
file10366    4
file11640    4
file10467    4
file12934    3
file10093    3
file480      3
file1409     3
file12356    3
file8531     3
file4407     3
file3188     3
file7061     3
file5631     3
file11380    3
file10438    3
file4397     3
file11280    3
file1151     3
file11366    3
file11424    3
file1774     3
file1333     2
file945      2
file2265     2
file11505    2
file3775     2
file12295    2
file9380     2
file7614     2
file227      2
file4361     2
file875      2
file10389    2
file1591     2
file6891     2
file9050     2
file1450     2
file9125     2
file6459     1
file6224     1
file1206     1
file8485     1
Name: count, dtype: int64

In [22]:
# Save the lloom to a pickle file
l.save(folder="/Users/lenap/Desktop", file_name="lloom_1")

Saved session to /Users/lenap/Desktop/lloom_1.pkl


## v2: Auto mode

LLooM also provides a one-function **auto** mode that grants less control, but simplifies the generation and scoring process into a single function. You can try out this version with the functions below.

### Create a LLooM instance
Then, after loading your data as a Pandas DataFrame, create a new LLooM instance. You will need to specify the name of the column that contains your input text documents (`text_col`). The ID column (`id_col`) is optional.

In [ ]:
# Set up the LLooM instance with the specified dataset
l = wb.lloom(
    df=df,
    text_col="text",
    id_col="doc_id",  # Optional
)

### Run concept generation
Next, you can go ahead and start the concept induction process by generating concepts. You can omit the `seed` parameter if you do not want to use a seed.

In [ ]:
cur_seed = None  # Optionally replace with string
score_df = await l.gen_auto(seed=cur_seed, max_concepts=5)

In [ ]:
# View cost/time summary
l.summary()

### Visualize results
Now, you can visualize the results in the main **LLooM Workbench** view. An interactive widget will appear when you run the `vis` function:
![LLooM Workbench UI](https://github.com/michelle123lam/lloom/blob/main/docs/public/media/lloom_workbench_ui.png?raw=1)

The **Concept Overview (A)** provides a high-level summary. Click on a concept row in the **Concept Matrix (B)** to see its **Detail View (C)**, or click on a slice column to see its corresponding Detail View.

In [ ]:
# Visualize concept results
# Group data by the number of likes (automatically binned) with slice_col
l.vis(slice_col="Likes")

In [ ]:
# Visualize concept results
# Group data by page category with slice_col
l.vis(slice_col="Page Category")

### (Optional) Try normalizing by slice or by concept


In [ ]:
l.vis(slice_col="Likes", norm_by="slice")

In [ ]:
l.vis(slice_col="Likes", norm_by="concept")

### (Optional) Add manual concept
You may also manually add your own custom concepts by providing a name and prompt. This will automatically score the data by that concept. Re-run the `vis()` function to see the new concept results.

In [ ]:

# Add a custom concept with the given name and prompt
await l.add(
    name="Your new concept name",
    prompt="Your new concept criteria prompt",  # Ex: "Does the text include [...]?"
)

In [ ]:
# Visualize concept results
l.vis(slice_col="Likes")

### (Optional) Submit your results
**🖼️ ✨ Submit your work for a chance to be featured on our site!**

If you'd like to share what you've done with LLooM or would like your work featured in a gallery of results, please submit your LLooM instance with the `submit()` function! If your submission is selected, we'll reach out to you to follow up and hear more about your work with LLooM.

In [ ]:
l.submit()  # You will be prompted to provide a few details about your analysis

### (Optional) Export and/or save results

In [ ]:
# Export the results to a dataframe
export_df = l.export_df()

In [ ]:
export_df.head()

In [ ]:
# Save the lloom to a pickle file
l.save(folder="your/path/here", file_name="your_file_name")